In [9]:
from qiskit import QuantumCircuit

def build_iqft(n):
    qc = QuantumCircuit(n)

    for target in reversed(range(n)):
        for control in reversed(range(target + 1, n)):
            k = control - target + 1
            angle = -2 * np.pi / (2**k)
            qc.cp(angle, control, target)
        
        qc.h(target)
        qc.barrier()

    for i in range(n // 2):
        qc.swap(i, n - i - 1)
        
    return qc

In [10]:
from qiskit_aer import AerSimulator
from qiskit import transpile
%run ../Quantum_Fourier_Transformation/Quantum_Fourier_Transformation.ipynb

n = 3
qc = QuantumCircuit(n, n)

qc.x(0)
qc.x(1)
qc.barrier()

qft = build_qft(n).to_instruction()
qc.append(qft, range(n))

iqft = build_iqft(n).to_instruction()
qc.append(iqft, range(n))

qc.measure(range(n), range(n))
simulator = AerSimulator()
compiled_qc = transpile(qc, simulator)
result = simulator.run(compiled_qc, shots=1024).result()
counts = result.get_counts()
print(f"Counts: {counts}")

                             ░                ░ ┌───┐ ░    
q_0: ──────■─────────────────░───────■────────░─┤ H ├─░──X─
           │                 ░ ┌───┐ │P(π/2)  ░ └───┘ ░  │ 
q_1: ──────┼────────■────────░─┤ H ├─■────────░───────░──┼─
     ┌───┐ │P(π/4)  │P(π/2)  ░ └───┘          ░       ░  │ 
q_2: ┤ H ├─■────────■────────░────────────────░───────░──X─
     └───┘                   ░                ░       ░    
State |000> (0): Amplitude = 0.354, Phase = 0.0°
State |001> (1): Amplitude = 0.354, Phase = 135.0°
State |010> (2): Amplitude = 0.354, Phase = 270.0°
State |011> (3): Amplitude = 0.354, Phase = 45.0°
State |100> (4): Amplitude = 0.354, Phase = 180.0°
State |101> (5): Amplitude = 0.354, Phase = 315.0°
State |110> (6): Amplitude = 0.354, Phase = 90.0°
State |111> (7): Amplitude = 0.354, Phase = 225.0°
Counts: {'011': 1024}
